<h3><b style="color:red">IMPORTANT:</b> Restart your session everytime you open this workshop</h3>

In [ ]:
%pip install smolagents langchain_community

# Tools

This notebook contains all the tools that will be used by the agent.

In [1]:
import requests
import sqlite3
import pandas as pd

from smolagents import tool

from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

/tmp/ipykernel_13562/869068460.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun


## Creating simple tool

A Python function must be annotated with `@tool`. It should also have a docstring describing what does the function do, what does it return and the description of its parameters.

### City to location

The following function look up at latitude and longitude of a city.

In [2]:
# TODO: Load CSV file containing latitude, longitude and altitude of cities
# https://github.com/bahar/WorldCityLocations/tree/master
df = pd.read_csv('/content/data/cities_latlng.csv', sep=';')


In [3]:
# TODO: Explore the loaded dataframe
df.head()

,id,country,city,latitude,longitude,altitude
0,1,Afghanistan,Kabul,34.516667,69.183334,1808
1,2,Afghanistan,Kandahar,31.610000,65.699997,1015
2,3,Afghanistan,Mazar-e Sharif,36.706944,67.112221,369
3,4,Afghanistan,Herat,34.340000,62.189999,927
4,5,Afghanistan,Jalalabad,34.420000,70.449997,573


In [13]:
# TODO: Add tool description

@tool
def get_latlng(city: str) -> any:
  """
  Get the altitude, latitude and longitude of a city.

  Args:
    city: The name of the city.

  Returns:
    A dictionary containing the city name, altitude, latitude and longitude of the city.

  Example:
    get_latlng('Paris')

  """
  r = df.query(f"city.str.lower() == '{city.lower()}'")
  return { 'city': city, 'latitude': r.iloc[0]['latitude'], 'longitude': r.iloc[0]['longitude'], 'altitude': r.iloc[0]['altitude'] }

In [14]:
# TODO: Test get_latlng method
# case insensitive search
get_latlng('paris')

{'city': 'paris',
 'latitude': np.float64(48.85341),
 'longitude': np.float64(2.3487999),
 'altitude': np.int64(30)}

### Temperature at latitude and longitude

The following function lookup the weather at the given latitude and longtude.

In [17]:
# TODO: Add tool description
@tool
def get_temperature(latitude: float, longitude: float) -> any:
  """
  Get the temperature at the given latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the temperature unit and the temperature.

  Example:
    get_temperature(48.8566, 2.3522)
  """

  url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
  resp = requests.get(url)
  j = resp.json()
  if resp.status_code >= 400:
     raise Exception(j['reason'])
  temperature = j['current']['temperature_2m']
  units = j['current_units']['temperature_2m']
  return { "temperature_unit": units, "temperature": temperature }

In [16]:
# TODO: Test get_temperature method
get_temperature(48.8566, 2.3522)

{'temperature_unit': '°C', 'temperature': 9.0}

### Query relational database

The following function queries a relational database (SQLite) view called `album_track`. The table's schema is as follows:
| Field name  | Type          |
|-------------|---------------|
| AlbumId     | integer       |
| Title       | nvarchar(160) |
| track_name  | nvarchar(200) |
| artist_name | nvarchar(120) |
| duration    | integer       |
| composer    | nvarchar(220) |


In [21]:
# TODO: Add tool description
@tool
def query_album_track(query: str) -> str:
  """
  Query the album_track table with SQL query. Returns the result as a array of tuples.
  The table name is `album_track`. The following is the table's schema:
    AlbumId: integer
    Title: nvarchar(160)
    track_name: nvarchar(200)
    artist_name: nvarchar(120)
    duration: integer
    composer: nvarchar(220)
  The duration is in milliseconds.

  Args:
    query: The SQL query in the SQLite dialect.

  Returns:
    The result of the query as a array of tuples.

  Example:
    query_album_track("SELECT * FROM album_track LIMIT 10")

  """
  database = "data/chinook_sqlite.sqlite"
  conn = sqlite3.connect(database)
  try:
     cursor = conn.cursor()
     rows = cursor.execute(query)
     return rows.fetchall()
  finally:
     conn.close()

In [20]:
# TODO: Test the query_album_track function
query_album_track("SELECT * FROM album_track LIMIT 10")

[(1,
  'For Those About To Rock We Salute You',
  'For Those About To Rock (We Salute You)',
  'AC/DC',
  343719,
  'Angus Young, Malcolm Young, Brian Johnson'),
 (2, 'Balls to the Wall', 'Balls to the Wall', 'Accept', 342562, None),
 (3,
  'Restless and Wild',
  'Fast As a Shark',
  'Accept',
  230619,
  'F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman'),
 (3,
  'Restless and Wild',
  'Restless and Wild',
  'Accept',
  252051,
  'F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman'),
 (3,
  'Restless and Wild',
  'Princess of the Dawn',
  'Accept',
  375418,
  'Deaffy & R.A. Smith-Diesel'),
 (1,
  'For Those About To Rock We Salute You',
  'Put The Finger On You',
  'AC/DC',
  205662,
  'Angus Young, Malcolm Young, Brian Johnson'),
 (1,
  'For Those About To Rock We Salute You',
  "Let's Get It Up",
  'AC/DC',
  233926,
  'Angus Young, Malcolm Young, Brian Johnson'),
 (1,
  'For Those About To Rock We Salute You',
  'Inject The Venom',
  'AC/DC',
  210834,
  'An